In [1]:
import duckdb

from config import (
    DB_PATH,
    JOINED_PARQUET,
    SENSORY_SUBSET_PARQUET,
    SENSORY_ANALYSIS_DIR,
    SENSORY_REPORT_MD,
)

In [2]:
COMPLAINT_TERMS = [
    "itchy",
    "scratchy",
    "rough",
    "uncomfortable",
    "tight",
    "too small",
    "too big",
    "thin",
    "cheap",
    "bad quality",
    "poor quality",
    "seam",
    "tag",
    "irritating",
    "hot",
    "heavy",
    "stiff",
    "shrunk",
    "shrink",
    "ripped",
    "tore",
    "fell apart",
]

In [3]:
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def fetch_markdown_table(con, query):
    df = con.execute(query).fetchdf()
    return df.to_markdown(index=False)


In [4]:
def main():
    SENSORY_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

    print("Connecting to DuckDB:")
    print(DB_PATH)

    con = duckdb.connect(str(DB_PATH))

    print("\nLoading full and sensory datasets...")

    con.execute(
        """
        CREATE OR REPLACE TABLE joined_reviews AS
        SELECT *
        FROM read_parquet(?)
        """,
        [str(JOINED_PARQUET)],
    )

    con.execute(
        """
        CREATE OR REPLACE TABLE sensory_reviews AS
        SELECT *
        FROM read_parquet(?)
        """,
        [str(SENSORY_SUBSET_PARQUET)],
    )

    complaint_selects = []

    for term in COMPLAINT_TERMS:
        escaped = sql_escape(term)
        complaint_selects.append(
            f"""
            SELECT
                '{escaped}' AS complaint_term,
                COUNT(*) AS mention_count
            FROM sensory_reviews
            WHERE rating <= 2
              AND LOWER(COALESCE(review_text, '')) LIKE '%{escaped}%'
            """
        )
    
    complaint_query = "\nUNION ALL\n".join(complaint_selects)

    sections = {}

    sections["Dataset Comparison"] = fetch_markdown_table(
        con,
        """
        SELECT
            'Full Amazon Fashion' AS dataset,
            COUNT(*) AS review_rows,
            COUNT(DISTINCT parent_asin) AS products,
            COUNT(DISTINCT user_id) AS users,
            ROUND(AVG(rating), 2) AS avg_rating
        FROM joined_reviews

        UNION ALL

        SELECT
            'Sensory-related subset' AS dataset,
            COUNT(*) AS review_rows,
            COUNT(DISTINCT parent_asin) AS products,
            COUNT(DISTINCT user_id) AS users,
            ROUND(AVG(rating), 2) AS avg_rating
        FROM sensory_reviews
        """
    )

    sections["Sensory Rating Distribution"] = fetch_markdown_table(
        con,
        """
        SELECT
            rating,
            COUNT(*) AS review_count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM sensory_reviews
        GROUP BY rating
        ORDER BY rating
        """
    )

    sections["Top Sensory Stores"] = fetch_markdown_table(
        con,
        """
        SELECT
            store,
            COUNT(*) AS review_count,
            COUNT(DISTINCT parent_asin) AS product_count,
            ROUND(AVG(rating), 2) AS avg_rating
        FROM sensory_reviews
        WHERE store IS NOT NULL
          AND TRIM(store) != ''
        GROUP BY store
        ORDER BY review_count DESC
        LIMIT 20
        """
    )

    sections["Top Sensory Products"] = fetch_markdown_table(
        con,
        """
        SELECT
            product_title,
            store,
            COUNT(*) AS review_count,
            ROUND(AVG(rating), 2) AS avg_rating,
            ROUND(AVG(CASE WHEN rating <= 2 THEN 1 ELSE 0 END) * 100, 2) AS negative_review_pct
        FROM sensory_reviews
        GROUP BY product_title, store
        HAVING COUNT(*) >= 10
        ORDER BY review_count DESC
        LIMIT 20
        """
    )

    sections["Most Common Complaints in Low-Rated Sensory Reviews"] = fetch_markdown_table(
        con,
        f"""
        SELECT *
        FROM ({complaint_query})
        ORDER BY mention_count DESC
        """
    )

    sections["Low-Rated Review Examples"] = fetch_markdown_table(
        con,
        """
        SELECT
            product_title,
            store,
            rating,
            review_title,
            review_text
        FROM sensory_reviews
        WHERE rating <= 2
        LIMIT 20
        """
    )

    report = "# Sensory Clothing Analysis Report\n\n"

    report += (
        "This report focuses on Amazon Fashion reviews that mention sensory-related "
        "clothing concerns such as softness, seams, tags, compression, itchiness, "
        "scratchiness, comfort, and texture.\n\n"
    )

    for title, table_md in sections.items():
        report += f"## {title}\n\n"
        report += table_md
        report += "\n\n"

    report += "## Initial Interpretation\n\n"
    report += (
        "- This subset helps narrow the full Amazon Fashion dataset toward products and reviews "
        "that are more relevant to sensory clothing concerns.\n"
        "- The main performance indicators to compare are rating distribution, average rating, "
        "negative review percentage, and review volume.\n"
        "- The complaint-term table gives a first-pass view of common issues, but later feature "
        "engineering should make this more systematic.\n"
        "- Product-level insights should be treated carefully because many products in the full "
        "dataset have very few reviews.\n"
    )

    with open(SENSORY_REPORT_MD, "w", encoding="utf-8") as f:
        f.write(report)

    print("\nSaved sensory analysis report to:")
    print(SENSORY_REPORT_MD)

    con.close()

In [5]:
if __name__ == "__main__":
    main()

Connecting to DuckDB:
/Users/IvyGuo/Documents/GitHub/AIL-sensory-fair/data/ail_sensory.duckdb

Loading full and sensory datasets...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Saved sensory analysis report to:
/Users/IvyGuo/Documents/GitHub/AIL-sensory-fair/outputs/sensory_analysis/Amazon_Fashion_sensory_analysis_report.md
